In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
REUSE_VESSELNESS = True
REBUILD_REPORT = True


# OpenPlaque — Global Left-Coronary Graph Reconstruction v1.1

Positive-control calibrated graph reconstruction. The validated ~25 mm LAD is the only positive coronary anchor. The graph must first reconnect detected nodes on the already accepted LAD before a negative topology result is interpretable. The validated RCA remains a hard 3 mm exclusion corridor and the aorta remains post-hoc only.


In [ ]:
!rm -rf /content/OpenPlaque
!git clone -q --depth 1 --branch lad-global-left-coronary-graph-from-main https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
%pip -q install numpy pandas scipy matplotlib SimpleITK scikit-image
import sys
sys.path.insert(0,'/content/OpenPlaque/src')
!git -C /content/OpenPlaque rev-parse HEAD


In [ ]:
from IPython.display import display, Image
from openplaque.lad_global_left_coronary_graph import synthetic_global_graph_self_test
from openplaque.lad_global_left_coronary_graph_v2 import GlobalLeftCoronaryGraphWorkflowV2, synthetic_global_graph_v2_self_test
test1=synthetic_global_graph_self_test(); test2=synthetic_global_graph_v2_self_test()
display(test1); display(test2)
assert test1['passed'], test1
assert test2['passed'], test2
wf=GlobalLeftCoronaryGraphWorkflowV2(root='/content/drive/MyDrive/OpenPlaque', reuse=REUSE_VESSELNESS)


In [ ]:
prov=wf.load_inputs(); display(prov)
print('Validated LAD length:', prov['lad_length_mm'])
print('CT cache:', prov['ct_cache'])


In [ ]:
wf.build_vesselness()
print('ROI source bounds:', wf.roi_lo, wf.roi_hi)
print('Isotropic vesselness shape:', wf.vessel.shape)


In [ ]:
nodes=wf.discover_nodes(); print('Graph nodes:', len(nodes)); display(nodes.sort_values('plane_score',ascending=False).head(30))
edges=wf.build_graph(); print('Graph edges:', len(edges)); display(edges.head(30))
print('Positive-control edge calibration:')
display(wf.edge_calibration)
print('Positive control passed:', wf.graph_positive_control_passed, 'adaptive edge radius mm:', wf.edge_radius_mm)


In [ ]:
paths=wf.enumerate_paths(); print('Novel proximal candidate graph paths:', len(paths)); display(paths)
summary=wf.validate_paths(); display(summary)
display(wf.validation)


In [ ]:
names=wf.make_figures()
for name in names:
    print(name)
    display(Image(filename=str(wf.out/name)))


In [ ]:
report,zip_path=wf.package()
print('STATUS:', wf.summary['status'])
print('POSITIVE CONTROL PASSED:', wf.summary.get('graph_positive_control_passed'))
print('ADAPTIVE EDGE RADIUS MM:', wf.summary.get('adaptive_edge_radius_mm'))
print('HTML:', report)
print('Final ZIP:', zip_path)
print('Drive search: https://drive.google.com/drive/u/0/search?q=OPENPLAQUE_LAD_GLOBAL_LEFT_CORONARY_GRAPH_REPORT_BACK.zip')
